# Drive test telemetry

Plots host-frame wheel position and velocity from `drive_test/telemetry.fthr`, written by `drive_test.cpp`.

Run this notebook with the `generic` conda env, after `make drive_test` from `backend/`. The Teensy already applies each motor's configured direction, so the traces use the same sign as the velocity command.

Motors are registered in the order `drive_center`, `drive_right`, `drive_left`, one channel per `RobParamCode`. Position and velocity are `f32`. Their time base is the `f64` `FEEDBACK_TS` channel, in monotonic seconds.

In [ ]:
import struct
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from feathersdk.quill_defs.python_qdefs.data_type_qdef import DataType
from feathersdk.quill_defs.python_qdefs.robstride_qdef import RobParamCode, rob_all_params

MOTORS = ("drive_center", "drive_right", "drive_left")
NPARAMS = len(rob_all_params)
HEADER_BYTES = 256
BLOCK_HEADER_BYTES = 128

_DTYPE_BYTES = {
    DataType.U8: 1, DataType.U16: 2, DataType.U32: 4, DataType.U64: 8,
    DataType.I8: 1, DataType.I16: 2, DataType.I32: 4, DataType.I64: 8,
    DataType.F32: 4, DataType.F64: 8, DataType.BOOL: 1,
}
ELEM_BYTES = [_DTYPE_BYTES[DataType(int(p.dtype))] for p in rob_all_params]

POSITION = int(RobParamCode.FB_POSITION)
VELOCITY = int(RobParamCode.FB_VELOCITY)
TIME = int(RobParamCode.FEEDBACK_TS)
KIND = {POSITION: "position", VELOCITY: "velocity", TIME: "time"}


def find_telemetry(explicit=None):
    """Locate drive_test/telemetry.fthr from this process's working directory."""
    if explicit:
        path = Path(explicit)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    relatives = (
        Path("drive_test/telemetry.fthr"),
        Path("tests/one_off_tests/drive_test/telemetry.fthr"),
        Path("feathersdk/src/feathersdk/backend/tests/one_off_tests/drive_test/telemetry.fthr"),
    )
    here = Path.cwd()
    for base in [here, *here.parents]:
        for rel in relatives:
            path = base / rel
            if path.is_file():
                return path
    raise FileNotFoundError(
        "drive_test/telemetry.fthr not found. Run make drive_test from backend/, "
        "or set TELEMETRY_PATH to the file."
    )


def iter_items(data):
    """Yield variable-format item payloads from a .fthr file.

    A short drive-test run is one block at offset 256. Each item is a uint64
    byte count followed by that many payload bytes. num_items == -1 means the
    block was still open; stop at the first item that does not fit.
    """
    if len(data) < HEADER_BYTES or data[:4] != b"FTHR":
        raise ValueError("not a .fthr file")
    file_format = data[44]
    chunk = struct.unpack_from("<Q", data, 45)[0]
    name = data[12:44].split(b"\0", 1)[0].decode()
    if file_format != 1:
        raise ValueError(f"{name}: expected variable format, got {file_format}")
    if chunk < BLOCK_HEADER_BYTES + 8:
        raise ValueError(f"{name}: chunk size {chunk} is too small")

    offset = HEADER_BYTES
    while offset + BLOCK_HEADER_BYTES <= len(data):
        num_items = struct.unpack_from("<q", data, offset)[0]
        pos = offset + BLOCK_HEADER_BYTES
        end = min(len(data), offset + chunk)
        count = 0
        while pos + 8 <= end and (num_items < 0 or count < num_items):
            (nbytes,) = struct.unpack_from("<Q", data, pos)
            pos += 8
            if nbytes > end - pos:
                break
            yield data[pos:pos + nbytes]
            pos += nbytes
            count += 1
        nxt = offset + chunk
        if nxt + BLOCK_HEADER_BYTES > len(data):
            break
        offset = nxt


def read_series(data):
    """Return {motor: {position, velocity, time}} lists, in sample order."""
    series = {name: {"position": [], "velocity": [], "time": []} for name in MOTORS}
    for blob in iter_items(data):
        if len(blob) < 2:
            continue
        (n_updates,) = struct.unpack_from("<H", blob, 0)
        pos = 2
        for _ in range(n_updates):
            if pos + 4 > len(blob):
                break
            tid, nvals = struct.unpack_from("<HH", blob, pos)
            pos += 4
            motor_i, pcode = divmod(tid, NPARAMS)
            if motor_i >= len(MOTORS):
                raise ValueError(f"telemetry id {tid} is outside the three drive motors")
            width = ELEM_BYTES[pcode]
            nbytes = width * nvals
            if pos + nbytes > len(blob):
                break
            raw = blob[pos:pos + nbytes]
            pos += nbytes
            kind = KIND.get(pcode)
            code = {4: "f", 8: "d"}.get(width)
            if kind is None or nvals == 0 or code is None:
                continue
            series[MOTORS[motor_i]][kind].extend(struct.unpack("<" + code * nvals, raw))
    return series


def file_info(data):
    name = data[12:44].split(b"\0", 1)[0].decode()
    closed = "clean" if data[53] == 1 else "torn"
    return name, closed


def plot_run(series):
    """Plot position and velocity against FEEDBACK_TS, in seconds from the first sample."""
    origins = [s["time"][0] for s in series.values() if s["time"]]
    if not origins:
        raise RuntimeError("telemetry file has no FEEDBACK_TS samples")
    t0 = min(origins)

    fig, axes = plt.subplots(2, 1, sharex=True, figsize=(10, 6))
    drawn = False
    for name, color in zip(MOTORS, ("tab:blue", "tab:orange", "tab:green")):
        times = series[name]["time"]
        for ax, kind, ylabel in (
            (axes[0], "position", "position (rad)"),
            (axes[1], "velocity", "velocity (rad/s)"),
        ):
            values = series[name][kind]
            n = min(len(values), len(times))
            if len(values) != len(times):
                print(f"{name} {kind}: {len(values)} samples, {len(times)} timestamps; plotting {n}")
            if n == 0:
                continue
            t = np.asarray(times[:n]) - t0
            ax.plot(t, values[:n], label=name, color=color)
            drawn = True
            ax.set_ylabel(ylabel)
            ax.grid(True, alpha=0.3)
        n_pos = min(len(series[name]["position"]), len(times))
        if n_pos:
            span = times[n_pos - 1] - times[0]
            print(f"{name}: {n_pos} samples over {span:.3f} s")

    if not drawn:
        raise RuntimeError("telemetry file has no position or velocity samples")
    axes[1].set_xlabel("time (s)")
    axes[0].set_title("Drive wheels")
    axes[0].legend(loc="best")
    fig.tight_layout()
    plt.show()


In [ ]:
# Set this to a specific file, or leave None to find drive_test/telemetry.fthr.
TELEMETRY_PATH = None

path = find_telemetry(TELEMETRY_PATH)
data = path.read_bytes()
name, closed = file_info(data)
print(f"{path}  ({name}, closed {closed}, {len(data)} bytes)")
plot_run(read_series(data))
